Data Collection: Gather data from given sources (databases, files, APIs, etc.).

Key Actions:

Identify and access relevant data sources. Ensure data formats are compatible with processing tools.

Output: data collected from the given link which are in the structured CSV format

In [1]:
import pandas as pd
import numpy as np
import os   # to see the path of current working directory

In [ ]:
print(os.getcwd())  # disply current path

In [3]:
# importing and reading a file

# reading a csv file as a dataframe
# df = pd.read_csv('../data/raw/IBM_Telco_Customer_Churn.csv', sep=',', encoding='latin1')
df = pd.read_excel('../data/raw/IBM-Telco-Customer-Churn.xlsx')

In [12]:
print("data size is:", df.shape)
print(df.head())

data size is: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMo

In [9]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [16]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


In [ ]:
# includes all data types
df.describe(include='all').T

Section 2: Data Integrity & The Detective Phase
Here will hunt for missing data, hidden formatting errors, and logic discrepancies.

Case Alpha: Hunting the Hidden Blanks
As we discussed, TotalCharges is read as an object (text) because of empty spaces. Let's find them:

In [19]:
# Locate rows where TotalCharges is just a blank space string
blank_charges = df[df['TotalCharges'].str.strip() == '']
print(f"Number of rows with hidden blank spaces in TotalCharges: {len(blank_charges)}")

# Inspect the tenure of these blank rows to find the underlying business logic
print(blank_charges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

Number of rows with hidden blank spaces in TotalCharges: 11
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             


"Data Investigation revealed 11 rows where TotalCharges contained empty string characters rather than traditional null values. Cross-referencing showed that all 11 records had a tenure of 0 months. This represents brand-new subscribers who have not yet completed their initial billing lifecycle."

Case Beta: Checking Duplicates & Constraints

In [20]:
# Verify if CustomerID is truly unique across all records
duplicate_count = df['customerID'].duplicated().sum()
print(f"Duplicate Customer IDs found: {duplicate_count}")

Duplicate Customer IDs found: 0


There are no duplicate data in the dataset

Section 3: Algorithmic Cleaning & Transformation

Now that you have mapped the data gaps, apply programmatic transformations to build clean layers for your downstream Power BI Star Schema.

In [21]:
# 1. Handle the 0-month tenure anomaly
# Replace blank spaces with '0' and cast the column to a proper numeric float type
df['TotalCharges'] = df['TotalCharges'].replace(' ', '0')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# 2. Standardize Clean Categorical Flags
# Standardize uniform boolean indicators for cleaner Power BI slicing
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# 3. Double-check your cleansing work
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].dtypes)

tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object


Section 4: Exporting the Clean Star-Schema Data Layers
To keep our portfolio professional, write a script to partition this single flat dataframe out into clean, separate files matching our target Star Schema layout.

In [25]:
# Create Dim_Customer
dim_customer = df[['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']].copy()

# Create Dim_Services
dim_services = df[['customerID', 'PhoneService', 'MultipleLines', 'InternetService', 
                   'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                   'TechSupport', 'StreamingTV', 'StreamingMovies']].copy()

# Create Fact_Subscriber_Billing
fact_billing = df[['customerID', 'tenure', 'Contract', 'PaperlessBilling', 
                  'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']].copy()

# Export clean, modular states directly to your project's clean data directory
dim_customer.to_csv('../data/processed/dim_customer.csv', index=False)
dim_services.to_csv('../data/processed/dim_services.csv', index=False)
fact_billing.to_csv('../data/processed/fact_subscriber_billing.csv', index=False)

print("Data pipeline executed successfully! Cleaned dimensional tables exported to /data/processed/")

Data pipeline executed successfully! Cleaned dimensional tables exported to /data/processed/
